# Reading Volcano and MA Plots (R)

_BioNexus Hub - bionexushub.com · Companion to the lesson._

**Set the runtime to R first:** Runtime, then Change runtime type, then R.


## 1. Set up and get a results table


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("DESeq2","apeglm"), update = FALSE, ask = FALSE)
library(DESeq2)


In [ ]:
base <- "https://raw.githubusercontent.com/owkin/PyDESeq2/main/datasets/synthetic/"
counts <- as.matrix(read.csv(paste0(base,"test_counts.csv"), row.names=1))
metadata <- read.csv(paste0(base,"test_metadata.csv"), row.names=1)
metadata$condition <- factor(metadata$condition)
metadata <- metadata[colnames(counts), , drop=FALSE]
dds <- DESeqDataSetFromMatrix(counts, metadata, design = ~ condition)
dds <- DESeq(dds)
res <- results(dds, contrast=c("condition","B","A"))
res_shrunk <- lfcShrink(dds, coef="condition_B_vs_A", type="apeglm")


## 2. MA plot (the health check)
Use the shrunken results so low-count noise does not spray dots everywhere.


In [ ]:
plotMA(res_shrunk, ylim=c(-3,3), main="MA plot (shrunken)")


## 3. Volcano plot (the hit finder)


In [ ]:
df <- as.data.frame(res)
df <- df[!is.na(df$padj), ]
df$sig <- df$padj < 0.05 & abs(df$log2FoldChange) > 1

plot(df$log2FoldChange, -log10(df$padj), pch=20,
     col=ifelse(df$sig, "#a855f7", "#cbd5e1"),
     xlab="log2 fold change (B vs A)", ylab="-log10 adjusted p-value",
     main="Volcano plot")
abline(h=-log10(0.05), lty=2, col="#64748b")
abline(v=c(-1,1), lty=2, col="#64748b")

top <- head(df[order(df$padj), ], 5)
text(top$log2FoldChange, -log10(top$padj), labels=rownames(top), pos=3, cex=0.7)


### Level up
For a publication-ready figure, try the EnhancedVolcano package: BiocManager::install("EnhancedVolcano").
